# NB_Load_Master_Data
Enterprise Banking Fabric Bronze Load

Author: Rakesh Soma

In [ ]:
from pyspark.sql import SparkSession
from datetime import datetime
import traceback

spark = SparkSession.builder.appName("EnterpriseBankingFabric").getOrCreate()

SQL_SERVER="RAKESH"
DATABASE="BankingERP"
USERNAME="Fabric_ETL_User"
PASSWORD="<KEYVAULT_SECRET>"

jdbc_url=f"jdbc:sqlserver://{SQL_SERVER}:1433;databaseName={DATABASE};encrypt=false"
connection_properties={
    "user":USERNAME,
    "password":PASSWORD,
    "driver":"com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

master_tables=[
"CountryMaster",
"CurrencyMaster",
"BranchMaster",
"CustomerTypeMaster",
"RiskCategoryMaster",
"OccupationMaster",
"AccountTypeMaster",
"LoanTypeMaster",
"CardTypeMaster",
"TransactionTypeMaster"
]

load_results=[]

print("="*60)
print("Enterprise Banking Bronze Load Started")
print(datetime.now())
print("="*60)

for table in master_tables:
    try:
        print(f"Loading Master.{table}")
        df=spark.read.jdbc(
            url=jdbc_url,
            table=f"Master.{table}",
            properties=connection_properties
        )
        row_count=df.count()
        df.write.mode("overwrite").format("delta").saveAsTable(f"Bronze.{table}")
        load_results.append((table,row_count,"SUCCESS"))
        print(f"{table} loaded successfully ({row_count} rows)")
    except Exception as e:
        load_results.append((table,0,"FAILED"))
        print(f"Failed to load {table}")
        print(traceback.format_exc())

print("\nSummary")
for r in load_results:
    print(r)

print("="*60)
print("Notebook Completed")
print(datetime.now())
print("="*60)
